## DL Hackathon Starter Colab
### Date: April 27, 2026
### DSBA+ICEF, HSE University

<small><font color=gray>Notebook authors: <a href="https://www.hse.ru/en/staff/aboldyrev" target="_blank">Alexey Boldyrev</a>, <a href="www.hse.ru/en/staff/mekarpov" target="_blank">Maksim Karpov</a>, <a href="https://www.hse.ru/en/staff/sara" target="_blank">Saraa Ali</a>, <a href="http://wiki.cs.hse.ru/Deep_Learning_DSBA_2025/2026" target="_blank">Stanislav Ryazanov</a>.

[**Instructions**](https://colab.research.google.com/drive/1owkYjuRGkx050LQnM3b3yTzd0Dr2XbeV) for running Colabs.

## Problem Description

**Motivation**: One of the most valuable sources of customer information is bank transaction data. In this set, there are many answers to the question: is it possible to predict the gender of a client using information about receipt and payment by bank card? And if so, what is the accuracy of such a prediction?

**Task**: Predict ROC AUC from the probability of gender (0|1) for each `cid` (customer ID) which is missing a gender in the file `gender.csv`.

<small>**CONSENT.** <mark>[ X ]</mark> We consent to sharing our Colab (after the Hackathon ends) with other students/instructors for educational purposes.

In [2]:
from google.colab import userdata

kaggle_token = userdata.get('kaggle_api')

In [4]:
!mkdir -p ~/.kaggle                                      # .kaggle folder must contain kaggle.json for kaggle executable to properly authenticate you to Kaggle.com
!echo "$kaggle_token" > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token
!kaggle config set -n competition -v hse-dl-hackathon-2026          # hackathon dataset
!kaggle competitions download -c hse-dl-hackathon-2026 >> log                                # download competition dataset as a zip file
!unzip -o *.zip >> log                                              # kaggle dataset is copied as a single file and needs to be unzipped
!kaggle competitions leaderboard -c hse-dl-hackathon-2026 --show                             # print public leaderboard

- competition is now set to: hse-dl-hackathon-2026
100% 35.7M/35.7M [00:04<00:00, 9.16MB/s]
Next Page Token = CfDJ8CS0IeAoHcJGgSEc27rBbk4mciSpFNq_CS3F3_ta_8MqQx48iGD5iq3rTEJNnzeLTRS4zpnj-u4bgYFhHO_9-zE
  teamId  teamName           submissionDate              score    
--------  -----------------  --------------------------  -------  
15748186  AC                 2026-04-27 07:11:42.940000  0.88537  
15748323  Team O             2026-04-27 07:01:07.266000  0.88403  
15748318  Maksim Rodikov     2026-04-27 07:09:39         0.88278  
15748331  Team R             2026-04-27 07:19:31.903000  0.88255  
15748260  Nikita Podlednev   2026-04-27 07:18:40.333000  0.88254  
15748191  W                  2026-04-27 07:16:04.740000  0.88237  
15748248  Team M             2026-04-27 07:16:02.206000  0.88179  
15748327  AO-Gazprom         2026-04-27 07:19:30.113000  0.88159  
15748408  K                  2026-04-27 07:19:49.936000  0.88134  
15748167  G_team             2026-04-27 06:59:05.973000  0.88

In [5]:
import os, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0)
print(f'Device: {DEVICE}')

pd.set_option('display.max_rows', 100, 'display.max_columns', 100, 'display.max_colwidth', 100, 'display.precision', 2, 'display.max_rows', 4)

class Timer():
  def __init__(self, lim:'RunTimeLimit'=200): self.t0, self.lim, _ = time.time(), lim, print(f'⏳ started. You have {lim} sec. Good luck!')
  def ShowTime(self):
    msg = f'Runtime is {time.time()-self.t0:.0f} sec'
    print(f'\033[91m\033[1m' + msg + f' > {self.lim} sec limit!!!\033[0m' if (time.time()-self.t0-1) > self.lim else msg)

# Set all random numbers to ensure that your private LB score for Kaggle is reproducible with IPYNB file, which you submit via LMS.
def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # When running on the CuDNN backend, two further options must be set
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Set a fixed value for the hash seed
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"Random seed set as {seed}")

Device: cuda


### Load data

In [6]:
gender = pd.read_csv('gender.csv', index_col='cid')
dfTrx  = pd.read_csv('trx.csv')

tY = gender.dropna()                # labelled CIDs (train)
vY = gender[gender.gender.isna()]   # unlabelled CIDs (test)
print(f'transactions={len(dfTrx):,}  train CIDs={len(tY):,}  test CIDs={len(vY):,}')

transactions=3,749,578  train CIDs=6,400  test CIDs=2,000


1. **trx.csv**, a table with chronological transaction history for ~15 months. Each `cid` can have unequal number of transactions.
    * `cid`, identifier of a bank's client; a non-negative integer
    * `dt`, a date of the transaction; an whole number starting from 0
    * `mcc`, mcc code of the transaction; a whole number; a foreign key for `mcc` in *_mcc.csv* table
    * `ttc`, transaction's type; a whole number; a foreign key for `ttc` in *_ttc.csv* table
    * `amt`, sum of the transactions in some monetary units, rounded to integer. Positive/Negative values are credits/debits to the client's account.
    * `tid`, ID of the register terminal (point of sale or POS) where the transaction was made.
1. **gender.csv**, a table with a numeric `cid` and `gender` (0|1) columns. Predict gender for the rows missing a gender value (i.e. first few thousands). Use the remaining gender values as target values in training your model.
1. **_mcc.csv**, a lookup table with the unique [Merchant Category Codes](https://en.wikipedia.org/wiki/Merchant_category_code) (MCC) and their descriptions in Russian. These might be helpful in locating similar categories using keywords or semantics (for example with sentence vectors).
1. **_ttc.csv**, a lookup table with unique transaction type codes and their verbal descriptions in Russian.

In [7]:
tmr = Timer() # runtime limit (in seconds). Add all of your code after the timer

⏳ started. You have 200 sec. Good luck!


<hr color=red>

<font size=5>⏳</font> <strong><font color=orange size=5>Your Code, Documentation, Ideas and Timer - All Start Here...</font></strong>

Students: Keep all your definitions, code, documentation **between** ⏳ symbols. Modifying any code outside of the timed playground incurs penalties.

## Preprocessing Pipeline

Explain elements of your preprocessing pipeline i.e. feature engineering, subsampling, clustering, dimensionality reduction, etc.

### 1. Build per-CID features (Tier 1-3)

Using advanced feature engineering from the Sberbank Gender competition:
* **Tier 1:** MCC pivots (counts, sum of absolute amounts, and normalized shares).
* **Tier 2:** Amount statistics + Debit/Credit splits.
* **Tier 3:** Time (Day of Week) histograms, MCC Entropy, and explicitly hand-crafted Male/Female MCC priors.

In [8]:
all_cids = gender.index   # 0..8399

# Time and amount transformations
# `dt` is given as an integer day starting from 0, so we extract day of week
dfTrx['day'] = dfTrx['dt']
dfTrx['dow'] = dfTrx['day'] % 7
dfTrx['is_weekend'] = (dfTrx['dow'] >= 5).astype(int)
dfTrx['amount_abs'] = dfTrx['amt'].abs()
dfTrx['log_amount'] = np.log1p(dfTrx['amount_abs'])

print("Building Tier 1 features: MCC pivots...")
mcc_cnt = dfTrx.groupby(['cid','mcc']).size().unstack(fill_value=0).add_prefix('mcc_cnt_')
mcc_sum = dfTrx.groupby(['cid','mcc'])['amount_abs'].sum().unstack(fill_value=0).add_prefix('mcc_sum_')
mcc_share = mcc_sum.div(mcc_sum.sum(axis=1).replace(0, 1), axis=0).add_prefix('share_')

print("Building Tier 2 features: Amount stats + Debit/Credit splits...")
amt = dfTrx.groupby('cid')['amt'].agg(['sum','mean','std','median','min','max','count','skew',
        ('q05', lambda s: s.quantile(.05)), ('q95', lambda s: s.quantile(.95))])
deb = dfTrx[dfTrx['amt'] < 0].groupby('cid')['amount_abs'].agg(['sum','mean','count']).add_prefix('deb_')
cre = dfTrx[dfTrx['amt'] > 0].groupby('cid')['amt'].agg(['sum','mean','count']).add_prefix('cre_')

print("Building Tier 3 features: Time + Entropy + Gender priors...")
def entropy(s):
    p = s.value_counts(normalize=True).values
    return -(p * np.log(p + 1e-12)).sum()

mcc_stats = dfTrx.groupby('cid').agg(n_unique_mcc=('mcc','nunique'),
                                     mcc_entropy=('mcc', entropy))
dow_h = pd.crosstab(dfTrx['cid'], dfTrx['dow'], normalize='index').add_prefix('dow_')

# Hand-crafted gender MCC priors
FEMALE_MCC = [5977,5631,5621,5651,5641,5661,5691,5697,5944,5948,5992,7230,7297,7298]
MALE_MCC   = [5541,5542,5532,5533,5571,5734,5813,5921,5945,5993,7538,7941,7995,7997]
dfTrx['fem'] = dfTrx['mcc'].isin(FEMALE_MCC)
dfTrx['mal'] = dfTrx['mcc'].isin(MALE_MCC)

prior = dfTrx.groupby('cid').agg(fem_share=('fem','mean'), mal_share=('mal','mean'))
prior['fem_minus_mal'] = prior['fem_share'] - prior['mal_share']

print("Joining all features together...")
X_raw = (mcc_cnt.join(mcc_sum).join(mcc_share).join(amt).join(deb).join(cre)
     .join(mcc_stats).join(dow_h).join(prior).reindex(all_cids).fillna(0))

X_raw.columns = [str(c) for c in X_raw.columns] # Ensure all column names are strings
X = X_raw.astype('float32')

print(f'feature matrix: {X.shape}')

Building Tier 1 features: MCC pivots...
Building Tier 2 features: Amount stats + Debit/Credit splits...
Building Tier 3 features: Time + Entropy + Gender priors...
Joining all features together...
feature matrix: (8400, 580)


### 2. Train / validation split and standardization

In [9]:
X_train_full = X.loc[tY.index].values
y_train_full = tY['gender'].values.astype('float32')
X_test       = X.loc[vY.index].values

X_tr, X_va, y_tr, y_va = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=0, stratify=y_train_full
)

scaler = StandardScaler().fit(X_tr)
X_tr   = scaler.transform(X_tr).astype('float32')
X_va   = scaler.transform(X_va).astype('float32')
X_test = scaler.transform(X_test).astype('float32')

### 3. Define the model

Multilayer perceptron with two hidden layers with ReLU + dropout. Sigmoid is applied implicitly via `BCEWithLogitsLoss`.

In [10]:
HIDDEN  = 128
DROPOUT = 0.3

class MLP(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN, dropout=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

model = MLP(X_tr.shape[1]).to(DEVICE)
model

MLP(
  (net): Sequential(
    (0): Linear(in_features=580, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)

### 4. Training loop

In [11]:
EPOCHS = 30
BATCH  = 128
LR     = 1e-3

tr_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)),
    batch_size=BATCH, shuffle=True,
)
X_va_t = torch.from_numpy(X_va).to(DEVICE)

opt     = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
loss_fn = nn.BCEWithLogitsLoss()

best_auc = 0.0
best_state = None
for epoch in range(1, EPOCHS + 1):
    model.train()
    for xb, yb in tr_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        val_p = torch.sigmoid(model(X_va_t)).cpu().numpy()
    auc = roc_auc_score(y_va, val_p)
    if auc > best_auc:
        best_auc = auc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}  Val AUC: {auc:.4f}  (best {best_auc:.4f})')

print(f'\nBest Val AUC: {best_auc:.4f}')

Epoch   1  Val AUC: 0.8281  (best 0.8281)
Epoch   5  Val AUC: 0.8411  (best 0.8432)
Epoch  10  Val AUC: 0.8269  (best 0.8432)
Epoch  15  Val AUC: 0.8179  (best 0.8432)
Epoch  20  Val AUC: 0.8154  (best 0.8432)
Epoch  25  Val AUC: 0.8147  (best 0.8432)
Epoch  30  Val AUC: 0.8148  (best 0.8432)

Best Val AUC: 0.8432


### 5. Predict on test set and save submission

In [12]:
model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    test_p = torch.sigmoid(model(torch.from_numpy(X_test).to(DEVICE))).cpu().numpy()

submission = pd.DataFrame({'cid': vY.index, 'gender': test_p})
submission.to_csv('baseline_submission.csv', index=False, float_format='%.6f')
print(f'Saved baseline_submission.csv  ({len(submission)} rows)')
submission.head()

Saved baseline_submission.csv  (2000 rows)


,cid,gender
0,0,0.66
1,1,0.19
...,...,...
3,3,0.71
4,4,0.23


## **References:**

* Remember to cite your sources here! At the least, your [Dive into Deep Learning](https://d2l.ai/) textbook should be cited. Google Scholar allows you to effortlessly copy/paste an APA citation format for books and publications. Also cite StackOverflow, package documentation, and other meaningful internet resources to help your peers learn from these (and to avoid plagiarism claims).

1. ...
1. ...
1. ...

<font color=green><h4><b>* LLM Documentation if used</b></h4></font>


1. Model and Platform Information  
   - The full name, version and operation mode of the model (e.g., Qwen (Qwen3.6-35B-A3B), Claude Opus 4.7, Gemini 3.1 Pro (Thinking mode)).  
   - A link to the service or platform used (e.g., https://chat.openai.com, https://claude.ai).  
   - If applicable, specify the application or interface (e.g., browser version with built‑in assistant, Telegram bot, IDE extension, etc.).

2. Interaction Record  
   - Provide all prompts and the model’s responses in chronological order (e.g., Prompt 1 – Response 1; Prompt 2 – Response 2, etc.).  
   - Ensure that the full conversation relevant to your submission is preserved and clearly formatted.

3. Reflection  
   - Briefly evaluate the quality and usefulness of the AI’s contribution.  
   - Explain what specific problem or part of the task the LLM helped you address.  
   - State how you verified or modified the output to ensure correctness and originality.

4. Usage Limitations  
   - Using an LLM to generate a complete solution is strictly prohibited.
   - Using LLM to generate documentation on the use of LLM is also prohibited.
   - The tool may be used only for assistance (e.g., drafting, brainstorming, clarifying concepts), not for full problem‑solving or code generation.

5. Academic Integrity  
   - Any submission incorporating LLM‑generated material without the documentation described above will be considered an academic integrity violation and may be treated as plagiarism.  
   - The teaching team reserves the right to determine whether a submission shows signs of unacknowledged AI assistance.


## 💡**Starter Ideas**

1. Richer hand-crafted features (signed-amt, log-magnitude, calendar, category-richness, TTC histogram, MCC×sign cross)
1. 5-fold stratified CV with Out-of-Fold (OOF) predictions
1. LightGBM/CatBoost as a complementary ensemble member
1. OOF-tuned ensemble averaging
1. GRU/Transformer sequence model with embedded (`mcc`, `ttc`, weekday, sign, log|`amt`|-bucket) tokens
1. Self-supervised pre-training and pseudo-labelling using the unlabelled test CIDs
1. Polishing (multi-seed, cosine LR, label smoothing, pos_weight)

<font size=5>⌛</font> <strong><font color=orange size=5>Do not exceed competition's runtime limit!</font></strong>

<hr color=red>

In [13]:
tmr.ShowTime()    # measure Colab's runtime. Do not remove. Keep as the last cell in your notebook.

Runtime is 32 sec
